In [ ]:
CELL1:凯恩斯交叉图——存货调节机制。针对难点：直观理解为什么经济会向Y=PE靠拢，以及“非计划存货”的角色。

In [1]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider

@interact(Y_actual=FloatSlider(min=500, max=1500, step=50, value=1200, description='当前产出Y'),
          MPC=FloatSlider(min=0.5, max=0.9, step=0.05, value=0.8, description='MPC'))
def cell_1_keynesian_cross(Y_actual, MPC):
    Y = np.linspace(0, 1600, 100)
    G, T, I = 200, 200, 200
    PE = (MPC * (Y - T)) + I + G
    PE_actual = (MPC * (Y_actual - T)) + I + G
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=Y, y=Y, name='均衡线 (Y=PE)', line=dict(color='black', dash='dash')))
    fig.add_trace(go.Scatter(x=Y, y=PE, name='计划支出 (PE)', line=dict(color='blue', width=3)))
    
    # 绘制当前点和存货缺口
    inventory_change = Y_actual - PE_actual
    status = "存货积压 (减产)" if inventory_change > 0 else "存货枯竭 (增产)"
    
    fig.add_trace(go.Scatter(x=[Y_actual], y=[PE_actual], mode='markers+text', 
                             text=[f"实际支出<br>{status}"], textposition="bottom right",
                             marker=dict(color='red', size=12)))
    
    fig.update_layout(title=f"单元 1：凯恩斯交叉与存货调节 (存货变动: {inventory_change:.1f})",
                      xaxis_title="实际产出 Y", yaxis_title="计划支出 PE", template="plotly_white",
                      xaxis=dict(showline=True, linecolor='black'), yaxis=dict(showline=True, linecolor='black'))
    fig.show()

interactive(children=(FloatSlider(value=1200.0, description='当前产出Y', max=1500.0, min=500.0, step=50.0), FloatS…

In [ ]:
CELL2:乘数效应的动态放大。针对难点：理解为什么G增加1，收入增加不止1（无限几何级数的可视化）。

In [2]:
@interact(dG=FloatSlider(min=10, max=100, step=10, value=50, description='政府购买变动'),
          MPC=FloatSlider(min=0.5, max=0.9, step=0.05, value=0.75, description='MPC'))
def cell_2_multipliers(dG, MPC):
    rounds = np.arange(1, 11)
    # 计算每一轮的支出增量
    spend_per_round = [dG * (MPC**(i-1)) for i in rounds]
    total_increase = dG / (1 - MPC)
    
    fig = go.Figure(data=[go.Bar(x=rounds, y=spend_per_round, text=[f"{v:.1f}" for v in spend_per_round], 
                                 textposition='auto', marker_color='teal')])
    
    fig.update_layout(title=f"单元 2：支出乘数分解 (总收入增长预测: {total_increase:.1f})",
                      xaxis_title="传导轮次", yaxis_title="该轮新增收入", template="plotly_white",
                      xaxis=dict(tickmode='linear', showline=True), yaxis=dict(showline=True))
    fig.show()

interactive(children=(FloatSlider(value=50.0, description='政府购买变动', min=10.0, step=10.0), FloatSlider(value=0.…

In [ ]:
Cell 3：IS 曲线的推导——从利率到产出针对难点：建立r→I→PE→Y的传导链条。

In [3]:
from plotly.subplots import make_subplots
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider

@interact(r=FloatSlider(min=1, max=10, step=0.5, value=5, description='利率 r (%)'),
          MPC=FloatSlider(min=0.5, max=0.85, step=0.05, value=0.6, description='消费倾向'))
def cell_3_perfect_is_derivation(r, MPC):
    # 调整参数使数据落在 0-1000 的舒适视区
    # 计算公式：Y = [G - MPC*T + I_base - d*r] / (1 - MPC)
    G, T, I_base, d = 200, 200, 300, 15
    Y_range = np.linspace(0, 1200, 100)
    
    # 1. 计算当前均衡点
    I_current = I_base - d * r
    Y_star = (I_current + G - MPC * T) / (1 - MPC)
    PE_current = MPC * (Y_range - T) + I_current + G

    # 2. 生成完整的 IS 曲线数据用于底图对比
    r_full = np.linspace(1, 10, 20)
    Y_is_full = (I_base - d * r_full + G - MPC * T) / (1 - MPC)

    # 创建双面板图
    fig = make_subplots(rows=2, cols=1, 
                        subplot_titles=("1. 凯恩斯交叉：利率变动导致支出位移", "2. IS 曲线：记录所有均衡点的轨迹"),
                        vertical_spacing=0.15)

    # --- 上图：凯恩斯交叉 ---
    fig.add_trace(go.Scatter(x=Y_range, y=Y_range, name='Y=PE', line=dict(color='gray', dash='dot')), row=1, col=1)
    fig.add_trace(go.Scatter(x=Y_range, y=PE_current, name=f'PE (r={r}%)', line=dict(color='red', width=3)), row=1, col=1)
    # 标记当前交点
    fig.add_trace(go.Scatter(x=[Y_star], y=[Y_star], mode='markers+text', 
                             text=[f"均衡 Y={Y_star:.0f}"], textposition="top left",
                             marker=dict(size=12, color='black'), showlegend=False), row=1, col=1)

    # --- 下图：IS 曲线 ---
    # 绘制背景 IS 轨迹
    fig.add_trace(go.Scatter(x=Y_is_full, y=r_full, name='IS 轨迹', line=dict(color='red', dash='dash', width=1)), row=2, col=1)
    # 绘制当前点
    fig.add_trace(go.Scatter(x=[Y_star], y=[r], mode='markers+text',
                             text=[f"IS点 (Y:{Y_star:.0f}, r:{r}%)"], textposition="top right",
                             marker=dict(size=15, color='red', symbol='diamond')), row=2, col=1)

    # 绘制辅助虚线（连接上下图）
    fig.add_shape(type="line", x0=Y_star, y0=Y_star, x1=Y_star, y1=r, 
                  line=dict(color="RoyalBlue", width=2, dash="dash"), row="all", col=1)

    # 美化布局
    fig.update_layout(height=700, template="plotly_white", showlegend=False,
                      title_text=f"IS 曲线推导实验室 (当前乘数: {1/(1-MPC):.2f})")
    
    fig.update_xaxes(title_text="产出 Y", range=[0, 1200], showline=True, linecolor='black', row=1, col=1)
    fig.update_yaxes(title_text="计划支出 PE", range=[0, 1200], showline=True, linecolor='black', row=1, col=1)
    
    fig.update_xaxes(title_text="产出 Y", range=[0, 1200], showline=True, linecolor='black', row=2, col=1)
    fig.update_yaxes(title_text="利率 r (%)", range=[0, 12], showline=True, linecolor='black', row=2, col=1)

    fig.show()

interactive(children=(FloatSlider(value=5.0, description='利率 r (%)', max=10.0, min=1.0, step=0.5), FloatSlider…

In [ ]:
🧪 Cell 4：LM 曲线的深度推导实验室

In [4]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, FloatSlider

@interact(Y_input=FloatSlider(min=400, max=1200, step=50, value=800, description='收入水平 Y'),
          M_P=FloatSlider(min=200, max=500, step=25, value=300, description='货币供给 M/P'))
def cell_4_perfect_lm_derivation(Y_input, M_P):
    # 参数设定 (L = kY - hr)
    k, h = 0.5, 25
    r_range = np.linspace(0.1, 15, 100)
    Y_range = np.linspace(400, 1200, 100)
    
    # 1. 计算左图：当前货币需求曲线 (给定 Y_input)
    # M/P = kY - hr  => 这里的横轴是货币量，纵轴是 r
    # 我们画的是 L(r) 曲线，给定 Y_input
    money_demand_curve = k * Y_input - h * r_range
    
    # 计算当前均衡利率 r*
    # r = (kY - M/P) / h
    r_star = (k * Y_input - M_P) / h
    r_star = max(0.1, r_star) # 防止利率为负

    # 2. 计算右图：完整的 LM 轨迹 (用于对比)
    r_lm_full = (k * Y_range - M_P) / h

    # 创建左右布局
    fig = make_subplots(rows=1, cols=2, 
                        subplot_titles=("1. 货币市场：流动性偏好均衡", "2. LM 曲线：利率与收入的映射"),
                        horizontal_spacing=0.12)

    # --- 左图：货币市场 ---
    # 货币需求线
    fig.add_trace(go.Scatter(x=money_demand_curve, y=r_range, name='货币需求 L(r,Y)', 
                             line=dict(color='blue', width=3)), row=1, col=1)
    # 货币供给线 (垂直线)
    fig.add_vline(x=M_P, line_width=3, line_dash="dash", line_color="green", row=1, col=1)
    # 均衡点标记
    fig.add_trace(go.Scatter(x=[M_P], y=[r_star], mode='markers', 
                             marker=dict(size=12, color='black'), showlegend=False), row=1, col=1)

    # --- 右图：LM 曲线 ---
    # 绘制背景 LM 轨迹
    fig.add_trace(go.Scatter(x=Y_range, y=(k * Y_range - M_P) / h, name='LM 轨迹', 
                             line=dict(color='blue', dash='dash', width=1)), row=1, col=2)
    # 绘制当前 LM 点
    fig.add_trace(go.Scatter(x=[Y_input], y=[r_star], mode='markers+text',
                             text=[f"LM点 (Y:{Y_input:.0f}, r:{r_star:.1f}%)"], textposition="top left",
                             marker=dict(size=15, color='blue', symbol='square')), row=1, col=2)

    # --- 跨图辅助虚线 (连接均衡利率) ---
    fig.add_shape(type="line", x0=M_P, y0=r_star, x1=Y_input, y1=r_star,
                  xref="x1", yref="y1", xanchor="x2", yanchor="y2", # 这种写法较复杂，简化为直接连接
                  line=dict(color="gray", width=2, dash="dot"), row=1, col="all")

    # 美化布局
    fig.update_layout(height=500, template="plotly_white", showlegend=False,
                      title_text="LM 曲线推导实验室：货币需求随收入 Y 变动")
    
    fig.update_xaxes(title_text="实际货币余额 M/P", range=[0, 700], showline=True, linecolor='black', row=1, col=1)
    fig.update_yaxes(title_text="利率 r (%)", range=[0, 15], showline=True, linecolor='black', row=1, col=1)
    
    fig.update_xaxes(title_text="产出/收入 Y", range=[400, 1300], showline=True, linecolor='black', row=1, col=2)
    fig.update_yaxes(title_text="利率 r (%)", range=[0, 15], showline=True, linecolor='black', row=1, col=2)

    fig.show()

interactive(children=(FloatSlider(value=800.0, description='收入水平 Y', max=1200.0, min=400.0, step=50.0), FloatS…